# Predykcja i analiza błędów — sekwencja testowa `05`

Ładuje checkpoint z treningu (`train_long.ipynb`), inferencja na `CARLA_processed/05`, zapis do `pred_errors/05/`.

In [ ]:
import glob
import json
import os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
import torchvision.transforms as T
from tqdm.auto import tqdm

CONFIG = {
    "data_root": "../CARLA_processed",
    "test_sequence": "05",
    "checkpoint_path": "resnet_segmentation_goodidx.ckpt",
    "batch_size": 4,
    "num_workers": 0,
    "ignore_index": 0,
    "output_root": "pred_errors",
}

NUM_CLASSES = 13
CLASS_NAMES = (
    "Unlabeled",
    "Road",
    "RoadLine",
    "SideWalk",
    "Building",
    "Vegetation",
    "Sky",
    "Person",
    "Vehicle",
    "TwoWheeler",
    "TrafficObject",
    "Misc",
    "Water",
)


def build_label_lut(class_map: dict, size: int = 256) -> np.ndarray:
    lut = np.zeros(size, dtype=np.int64)
    for raw, idx in class_map.items():
        lut[int(raw)] = int(idx)
    return lut


class_map = {i: i for i in range(NUM_CLASSES)}
LABEL_LUT = build_label_lut(class_map)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
class CARLADataset(Dataset):
    def __init__(
        self,
        root: str,
        label_lut: np.ndarray,
        sequences: list[str] | None = None,
    ):
        self.samples = []
        self.label_lut = label_lut

        all_folders = sorted(
            f for f in os.listdir(root)
            if os.path.isdir(os.path.join(root, f)) and f.isdigit()
        )
        if sequences is not None:
            seq_set = set(sequences)
            all_folders = [f for f in all_folders if f in seq_set]

        for folder in all_folders:
            rgb_dir = os.path.join(root, folder, "rgb")
            mask_dir = os.path.join(root, folder, "segmentation_raw")
            if not os.path.isdir(rgb_dir) or not os.path.isdir(mask_dir):
                continue
            for img_path in sorted(glob.glob(os.path.join(rgb_dir, "*.png"))):
                name = os.path.basename(img_path)
                mask_path = os.path.join(mask_dir, name)
                if os.path.exists(mask_path):
                    self.samples.append((img_path, mask_path))

        self.img_tf = T.Compose([T.ToTensor()])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        mask = np.array(Image.open(mask_path))[:, :, 0]
        mask = self.label_lut[mask.astype(np.int64)]
        mask = torch.from_numpy(mask).long()
        img = self.img_tf(img)
        return img, mask, img_path


class SegModel(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        resnet = models.resnet34(weights=None)
        self.encoder = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1,
            resnet.layer2,
            resnet.layer3,
            resnet.layer4,
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, 2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 2, stride=2),
            nn.ReLU(),
            nn.Conv2d(16, num_classes, 1),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


seg_model = SegModel(num_classes=NUM_CLASSES).to(device)
_ckpt = torch.load(CONFIG["checkpoint_path"], map_location=device)
_sd = {
    k.replace("model.", "", 1): v
    for k, v in _ckpt["state_dict"].items()
    if k.startswith("model.")
}
seg_model.load_state_dict(_sd, strict=True)
seg_model.eval()
print("Załadowano:", CONFIG["checkpoint_path"])

In [ ]:
seq = CONFIG["test_sequence"]
out_dir = Path(CONFIG["output_root"]) / seq
pred_dir = out_dir / "prediction"
diff_dir = out_dir / "difference"
pred_dir.mkdir(parents=True, exist_ok=True)
diff_dir.mkdir(parents=True, exist_ok=True)

test_ds = CARLADataset(
    CONFIG["data_root"],
    LABEL_LUT,
    sequences=[seq],
)
test_loader = DataLoader(
    test_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
)
print(f"Sekwencja {seq}: {len(test_ds)} klatek → {out_dir}")

ignore = CONFIG["ignore_index"]
confusion = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
intersection = np.zeros(NUM_CLASSES, dtype=np.int64)
union = np.zeros(NUM_CLASSES, dtype=np.int64)
gt_pixels = np.zeros(NUM_CLASSES, dtype=np.int64)
wrong_pixels = np.zeros(NUM_CLASSES, dtype=np.int64)
total_valid = 0
total_correct = 0

with torch.no_grad():
    for imgs, masks, paths in tqdm(test_loader, desc="Inference"):
        imgs = imgs.to(device)
        masks_np = masks.numpy()
        logits = seg_model(imgs)
        logits = F.interpolate(
            logits,
            size=masks.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        pred = logits.argmax(dim=1).cpu().numpy().astype(np.uint8)

        for b in range(pred.shape[0]):
            name = os.path.basename(paths[b])
            gt = masks_np[b]
            pr = pred[b]
            diff = (pr != gt).astype(np.uint8) * 255

            Image.fromarray(pr, mode="L").save(pred_dir / name)
            Image.fromarray(diff, mode="L").save(diff_dir / name)

            valid = gt != ignore
            total_valid += int(valid.sum())
            total_correct += int((pr[valid] == gt[valid]).sum())

            for c in range(NUM_CLASSES):
                if c == ignore:
                    continue
                gt_c = gt == c
                pr_c = pr == c
                gt_pixels[c] += int(gt_c.sum())
                wrong_pixels[c] += int((gt_c & (pr != c)).sum())
                intersection[c] += int((gt_c & pr_c).sum())
                union[c] += int((gt_c | pr_c).sum())

            for g in range(NUM_CLASSES):
                for p in range(NUM_CLASSES):
                    confusion[g, p] += int(((gt == g) & (pr == p)).sum())

print("Zapisano maski prediction/ i difference/")

In [ ]:
def safe_div(a, b):
    return float(a / b) if b > 0 else None


per_class = []
ious = []
for c in range(NUM_CLASSES):
    if c == ignore:
        continue
    iou = safe_div(intersection[c], union[c])
    if iou is not None:
        ious.append(iou)
    per_class.append(
        {
            "class_id": c,
            "name": CLASS_NAMES[c],
            "iou": iou,
            "pixel_accuracy_on_gt": safe_div(intersection[c], gt_pixels[c]),
            "error_rate_on_gt": safe_div(wrong_pixels[c], gt_pixels[c]),
            "gt_pixels": int(gt_pixels[c]),
            "wrong_pixels": int(wrong_pixels[c]),
        }
    )

per_class_sorted = sorted(
    [x for x in per_class if x["iou"] is not None],
    key=lambda x: x["iou"],
)

summary = {
    "sequence": seq,
    "num_frames": len(test_ds),
    "checkpoint": CONFIG["checkpoint_path"],
    "ignore_index": ignore,
    "mean_iou": float(np.mean(ious)) if ious else None,
    "pixel_accuracy": safe_div(total_correct, total_valid),
    "per_class": per_class,
    "per_class_by_iou_ascending": [
        {"name": x["name"], "iou": x["iou"], "error_rate_on_gt": x["error_rate_on_gt"]}
        for x in per_class_sorted
    ],
    "confusion_matrix": confusion.tolist(),
}

metrics_path = out_dir / "metrics.json"
metrics_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"mIoU (bez klasy {ignore}): {summary['mean_iou']:.4f}" if summary["mean_iou"] else "mIoU: n/a")
print(f"Pixel accuracy (bez ignore): {summary['pixel_accuracy']:.4f}")
print("\nKlasy od najgorszego IoU (największy problem):")
for row in per_class_sorted:
    err = row["error_rate_on_gt"]
    err_s = f"{err:.2%}" if err is not None else "n/a"
    print(
        f"  [{row['class_id']:2d}] {row['name']:<14} IoU={row['iou']:.4f}  błąd na GT={err_s}  (GT px={row['gt_pixels']})"
    )
print(f"\nPełne metryki: {metrics_path}")